# AI as Judge

[G-Eval](https://deepeval.com/docs/metrics-llm-evals) is a framework that uses LLM as a judge to evaluate LLM outputs. The evaluation can be based on any criteria. G-Eval is implemented by a library called [DeepEval](https://deepeval.com/) which includes a broader set of tests.


In [1]:
%load_ext dotenv
%dotenv ../../05_src/.secrets

In [2]:
from openai import OpenAI
import os

document_folder = "../../05_src/documents/"
#blue_cross_file = "the_blue_cross.txt"
blue_cross_file = "chesterton.txt"
file_path = os.path.join(document_folder, blue_cross_file)

with open(file_path, "r", encoding="utf-8") as f:
    blue_cross_text = f.read()

In [3]:
instructions = "You are an helpful assistant that summarizes works of fiction with a quirky and bubbly approach."
PROMPT = """
    Summarize the following story in at most four paragraphs. Please include all key characters and plot points.
    <story>
    {story}
    </story>
    In addition to the summary, add an introduction paragraph where you greet the reader and a conclusion where you share an opinion about the story.
"""

In [4]:
import os
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})
response = client.responses.create(
    model="gpt-4o-mini",
    instructions=instructions,
    input=[
        {"role": "user", 
         "content": PROMPT.format(story=blue_cross_text)}
    ],
    temperature=1.2
)

In [5]:
response.output_text

"Hello, wonderful reader! Are you ready to dive into the delightful whimsical world of mystery and detection? Today, we’re frolicking through *The Innocence of Father Brown* by G.K. Chesterton, where our endearing little priest, Father Brown, teams up with the larger-than-life ex-thief, Flambeau, to untangle a series of intriguing conundrums that showcase human nature at its quirkiest and most complex. \n\nIn this charming collection of short stories, Father Brown employs his sharp intuition and keen understanding of human psychology to solve various mysteries. Each tale presents a unique puzzle, showcasing characters from delightful to dangerous, and cleverly weaves moral lessons into their escapades. The narratives engage readers with their clever plot twists and Chesterton’s trademark wit, all while examining the intricacies of right and wrong. \n\nFrom tracking down the fabulous (and tricky) jewel thief in “The Blue Cross” to unraveling the threads of crime and personal demons in “

# Answer Relevancy

The answer relevancy metric evaluates how relevant the actual output of the LLM app is compared to the provided input. This metric is self-explaining in the sense that the output includes a reason for the metric score.

The metric is calculated as:

$$
AnswerRelevancy=\frac{NumberRelevantStatements}{TotalStatements}
$$

Reference: [Answer Relevancy](https://deepeval.com/docs/metrics-answer-relevancy). 

In [6]:
!pip install deepeval


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel

model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    # api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

metric = AnswerRelevancyMetric(
    threshold=0.7,
    include_reason=True,
    model=model,
    
)

test_case = LLMTestCase(
    input=PROMPT.format(story=blue_cross_text),
    actual_output=response.output_text,
    
)

In [8]:
metric.measure(test_case)

c:\DSI\deploying-ai-env\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

1.0

In [9]:
from IPython.display import display, Markdown
display(Markdown(f'**Score**: {metric.score}'))
display(Markdown(f'**Reason**: {metric.reason}'))

**Score**: 1.0

**Reason**: The score is 1.00 because the response directly addresses the request for a summary of the story, including key characters and plot points, without any irrelevant statements.

# Other Metrics

Other useful metric functions include:

+ [Faithfulness](https://deepeval.com/docs/metrics-faithfulness): evaluates whether the `actual_output` factually aligns with the contents of  `retrieval_context`. 
+ [Contextual Precision](https://deepeval.com/docs/metrics-contextual-precision): evaluates whether nodes in your `retrieval_context` that are relevant to the given input are ranked higher than irrelevant ones. 
+ [Contextual Recall](https://deepeval.com/docs/metrics-contextual-recall): evaluates the extent of which the retrieval_context aligns with the expected_output. 
+ [Contextual Relevancy](https://deepeval.com/docs/metrics-contextual-relevancy): evaluates the overall relevance of the information presented in your retrieval_context for a given input. 

# G-Eval

[G-Eval](https://deepeval.com/docs/metrics-llm-evals) is a framework that uses LLM-as-a-judge with chain-of-thoughts (CoT) to evaluate LLM outputs based on ANY custom criteria. The G-Eval metric is the most versatile type of metric deepeval offers.

In [10]:
instructions = "You are an helpful assistant that specializes in works of fiction."
PROMPT = """
    Based on the story below, answer the question provided.
    <story>
    {story}
    </story>
    <question>
    Who is the main antagonist in the story and what motivates their actions?
    </question>
"""

In [11]:
response = client.responses.create(
    model="gpt-4o-mini",
    instructions=instructions,
    input=[
        {"role": "user", 
         "content": PROMPT.format(story=blue_cross_text)}
    ],
    temperature=0.7
)

In [12]:
response.output_text

'The main antagonist in "The Innocence of Father Brown" is Flambeau, who is portrayed as a master criminal. His motivation primarily stems from a desire for adventure, challenge, and the thrill of committing clever crimes. He seeks to outsmart the police and enjoys the artistry involved in his criminal exploits. \n\nHowever, as the stories progress, Flambeau\'s character evolves. He becomes more complex, ultimately showing a sense of honor and morality, especially in his interactions with Father Brown, who serves as a foil to his criminality. Flambeau\'s eventual desire to reform and seek redemption suggests a deeper internal conflict and a yearning for acceptance beyond his criminal persona. \n\nIn the stories, his actions are often driven by a combination of personal ambition, the allure of the chase, and a need to prove himself against the cleverness of Father Brown, whom he respects and admires.'

## Evaluation Criteria

The most straightforward way to establish a metric is by using a single criteria.

In [13]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

correctness_metric = GEval(
    name="Correctness",
    criteria="Determine whether the actual output is factually correct based on the context.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

In [14]:
test_case = LLMTestCase(
    input=PROMPT.format(story=blue_cross_text),
    actual_output=response.output_text
)
evaluate(test_cases=[test_case], metrics=[correctness_metric])

✨ You're running DeepEval's latest Correctness [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...



Metrics Summary

  - ✅ Correctness [GEval] (score: 0.7659939073248732, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The response accurately identifies Flambeau as the main antagonist and discusses his motivations, including his desire for adventure and challenge. It also notes his character evolution and complex relationship with Father Brown, which aligns with the story's themes. However, the response could be improved by providing more specific examples from the text to illustrate Flambeau's actions and motivations, as well as mentioning other potential antagonists or conflicts present in the stories., error: None)

For test case:

  - input: 
    Based on the story below, answer the question provided.
    <story>
    ﻿The Project Gutenberg eBook of The innocence of Father Brown
    
This ebook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give

⚠ WARNING: No hyperparameters logged.
» ]8;id=9902113;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 16.28s | token cost: 0.01665075 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Correctness [GEval]', threshold=0.5, success=True, score=0.7659939073248732, reason="The response accurately identifies Flambeau as the main antagonist and discusses his motivations, including his desire for adventure and challenge. It also notes his character evolution and complex relationship with Father Brown, which aligns with the story's themes. However, the response could be improved by providing more specific examples from the text to illustrate Flambeau's actions and motivations, as well as mentioning other potential antagonists or conflicts present in the stories.", strict_mode=False, evaluation_model='gpt-4o-mini', error=None, evaluation_cost=0.01665075, verbose_logs='Criteria:\nDetermine whether the actual output is factually correct based on the context. \n \nEvaluation Steps:\n[\n    "Identify the context of the input to understand the expected factual information.",\

## Evaluation Steps 

G-Eval is flexible in many ways: notice that we can establish an evaluation criteria or a set of evaluation steps, that can help in guiding the model to follow specific steps to perform the evaluation.

In [15]:
...

correctness_metric = GEval(
    name="Correctness",
    evaluation_steps=[
        "Check whether the facts in 'actual output' contradicts any facts in 'input'",
        "You should also heavily penalize omission of detail",
        "Vague language, or contradicting OPINIONS, are not OK"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

In [16]:
test_case = LLMTestCase(
    input=PROMPT.format(story=blue_cross_text),
    actual_output=response.output_text
)
result = evaluate(test_cases=[test_case], metrics=[correctness_metric])

✨ You're running DeepEval's latest Correctness [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...



Metrics Summary

  - ✅ Correctness [GEval] (score: 0.7591879112357837, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The response accurately identifies Flambeau as the main antagonist and discusses his motivations, including his desire for adventure and challenge. It also notes his character evolution and internal conflict, which aligns well with the complexity of the story. However, it could benefit from more specific references to particular stories or events that illustrate these motivations, which would strengthen the analysis., error: None)

For test case:

  - input: 
    Based on the story below, answer the question provided.
    <story>
    ﻿The Project Gutenberg eBook of The innocence of Father Brown
    
This ebook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License inc

⚠ WARNING: No hyperparameters logged.
» ]8;id=9902115;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 9.57s | token cost: 0.0165738 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [17]:
result.model_dump()

{'test_results': [{'name': 'test_case_0',
   'success': True,
   'metrics_data': [{'name': 'Correctness [GEval]',
     'threshold': 0.5,
     'success': True,
     'score': 0.7591879112357837,
     'reason': 'The response accurately identifies Flambeau as the main antagonist and discusses his motivations, including his desire for adventure and challenge. It also notes his character evolution and internal conflict, which aligns well with the complexity of the story. However, it could benefit from more specific references to particular stories or events that illustrate these motivations, which would strengthen the analysis.',
     'strict_mode': False,
     'evaluation_model': 'gpt-4o-mini',
     'error': None,
     'evaluation_cost': 0.0165738,
     'verbose_logs': 'Criteria:\nNone \n \nEvaluation Steps:\n[\n    "Check whether the facts in \'actual output\' contradicts any facts in \'input\'",\n    "You should also heavily penalize omission of detail",\n    "Vague language, or contradic